In [0]:
df = spark.table("workspace.default.churn_features")
display(df.limit(5))

CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Index,Gender_Index,Normalized_Balance,Normalized_EstimatedSalary
15634602,619,France,Female,42,2,0.0,1,1,1,101348.88,1,0.0,1.0,0.0,0.5067348931822989
15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2.0,1.0,0.334031478677259,0.5627087386845443
15619304,502,France,Female,42,8,159660.8,3,1,0,113931.57,1,0.0,1.0,0.6363571759354565,0.5696543519906151
15701354,699,France,Female,39,1,0.0,2,0,0,93826.63,0,0.0,1.0,0.0,0.4691200509648671
15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0,2.0,1.0,0.5002462155052676,0.3954003607344501


In [0]:
feature_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Geography_Index",
    "Gender_Index"
]

label_column = "Exited"

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_model = assembler.transform(df)

In [0]:
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_column
)

model = lr.fit(train_df)

In [0]:
coefficients = model.coefficients
intercept = model.intercept

feature_importance = list(zip(feature_columns, coefficients))

for feature, coef in feature_importance:
    print(feature, ":", coef)

print("Intercept:", intercept)

CreditScore : -0.0007005349013828589
Age : 0.07355091128701878
Tenure : -0.025187864358595758
Balance : 4.818176950759809e-06
NumOfProducts : -0.09472666151522718
HasCrCard : -0.0713843859875051
IsActiveMember : -1.1375195012085615
EstimatedSalary : 6.090555708862437e-07
Geography_Index : 0.10978122356298324
Gender_Index : 0.5177962504063673
Intercept: -3.913261253091288


In [0]:
predictions = model.transform(test_df)

display(predictions.select("Exited", "prediction", "probability").limit(10))

Exited,prediction,probability
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8653354483344233"",""0.13466455166557667""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9663043454017478"",""0.03369565459825219""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8775636383936498"",""0.12243636160635019""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7385064849703888"",""0.26149351502961116""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9402770884872288"",""0.059722911512771204""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.916916568176665"",""0.083083431823335""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7913155469192403"",""0.20868445308075967""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8653159380392216"",""0.13468406196077842""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9568959924898316"",""0.04310400751016841""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7659334590958778"",""0.23406654090412216""]}"


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def extract_prob(v):
    return float(v[1])

extract_udf = udf(extract_prob, DoubleType())

predictions = predictions.withColumn(
    "churn_probability",
    extract_udf("probability")
)

display(predictions.select("Exited", "churn_probability").limit(10))

Exited,churn_probability
0,0.13466455166557667
0,0.03369565459825219
0,0.12243636160635019
0,0.26149351502961116
0,0.059722911512771204
0,0.083083431823335
1,0.20868445308075967
0,0.13468406196077842
0,0.04310400751016841
0,0.23406654090412216


In [0]:
from pyspark.sql.functions import col, when, sum as _sum

metrics_df = predictions.select(
    col("Exited").alias("label"),
    col("prediction")
)

cm = metrics_df.select(
    _sum(when((col("label") == 1) & (col("prediction") == 1), 1).otherwise(0)).alias("TP"),
    _sum(when((col("label") == 0) & (col("prediction") == 0), 1).otherwise(0)).alias("TN"),
    _sum(when((col("label") == 0) & (col("prediction") == 1), 1).otherwise(0)).alias("FP"),
    _sum(when((col("label") == 1) & (col("prediction") == 0), 1).otherwise(0)).alias("FN")
)

cm.show()

+---+----+---+---+
| TP|  TN| FP| FN|
+---+----+---+---+
| 69|1468| 40|344|
+---+----+---+---+



In [0]:
row = cm.collect()[0]

TP = row["TP"]
TN = row["TN"]
FP = row["FP"]
FN = row["FN"]

print("TP:", TP, "TN:", TN, "FP:", FP, "FN:", FN)

TP: 69 TN: 1468 FP: 40 FN: 344


In [0]:
precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.8001041124414368
Precision: 0.6330275229357798
Recall: 0.16707021791767554
F1 Score: 0.26436781609195403


In [0]:
from pyspark.sql.functions import when, col

threshold = 0.3

predictions_custom = predictions.withColumn(
    "custom_prediction",
    when(col("churn_probability") >= threshold, 1).otherwise(0)
)

display(predictions_custom.select("Exited", "churn_probability", "custom_prediction").limit(10))

Exited,churn_probability,custom_prediction
0,0.13466455166557667,0
0,0.03369565459825219,0
0,0.12243636160635019,0
0,0.26149351502961116,0
0,0.059722911512771204,0
0,0.083083431823335,0
1,0.20868445308075967,0
0,0.13468406196077842,0
0,0.04310400751016841,0
0,0.23406654090412216,0


In [0]:
from pyspark.sql.functions import col, when, sum as _sum

metrics_df = predictions_custom.select(
    col("Exited").alias("label"),
    col("custom_prediction").alias("prediction")
)

cm = metrics_df.select(
    _sum(when((col("label") == 1) & (col("prediction") == 1), 1).otherwise(0)).alias("TP"),
    _sum(when((col("label") == 0) & (col("prediction") == 0), 1).otherwise(0)).alias("TN"),
    _sum(when((col("label") == 0) & (col("prediction") == 1), 1).otherwise(0)).alias("FP"),
    _sum(when((col("label") == 1) & (col("prediction") == 0), 1).otherwise(0)).alias("FN")
)

cm.show()

+---+----+---+---+
| TP|  TN| FP| FN|
+---+----+---+---+
|182|1294|214|231|
+---+----+---+---+



In [0]:
row = cm.collect()[0]

TP = row["TP"]
TN = row["TN"]
FP = row["FP"]
FN = row["FN"]

print("TP:", TP, "TN:", TN, "FP:", FP, "FN:", FN)

TP: 182 TN: 1294 FP: 214 FN: 231


In [0]:
precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.7683498178032275
Precision: 0.4595959595959596
Recall: 0.4406779661016949
F1 Score: 0.449938195302843


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="Exited",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)
print("AUC:", auc)

AUC: 0.7337187943558514


In [0]:
high_risk = (
    predictions
    .select("CustomerId", "churn_probability", "Balance", "IsActiveMember")
    .orderBy("churn_probability", ascending=False)
    .limit(100)
)
display(high_risk)

CustomerId,churn_probability,Balance,IsActiveMember
15737647,0.8994983425990614,135120.56,0
15678916,0.8349043361279095,0.0,0
15673995,0.810801687459201,102541.1,0
15623314,0.8057392116631209,190353.08,0
15717879,0.7913460877441308,108078.56,1
15679249,0.7830187091451412,163146.46,0
15766575,0.7814406717246565,140745.33,0
15727353,0.7717104489646469,142028.36,0
15760244,0.7697617337431257,160979.68,1
15589572,0.7555678998362283,129855.72,0


In [0]:
full_predictions = model.transform(df_model)

full_predictions = full_predictions.withColumn(
    "churn_probability",
    extract_udf("probability")
)

high_risk = (
    full_predictions
    .select("CustomerId", "churn_probability", "Balance", "IsActiveMember")
    .orderBy("churn_probability", ascending=False)
    .limit(100)
)

display(high_risk)

CustomerId,churn_probability,Balance,IsActiveMember
15653251,0.9392862919984958,87873.39,0
15737647,0.8994983425990614,135120.56,0
15653050,0.875210145605868,95052.29,0
15594391,0.8702378693715802,183555.24,0
15655360,0.8685277828279938,148666.99,0
15790113,0.8586938988373507,113317.1,0
15764927,0.848475260107356,121513.31,1
15591107,0.8398608321066796,110357.0,0
15678916,0.8349043361279095,0.0,0
15648967,0.8332647469666821,169362.43,0
